# Implement Multi-Query Attention from Scratch
### Problem Statement
Multi-Query Attention (MQA), introduced in [Shazeer 2019](https://arxiv.org/abs/1911.02150), is a memory-efficient variant of Multi-Head Attention. Instead of having separate K and V projections per head, MQA uses a **single shared K/V head** across all query heads.

This dramatically reduces the KV cache size during autoregressive inference (from `num_heads × d_head` to just `1 × d_head` per token), making it much faster for long-sequence generation.

Your goal is to implement MQA from scratch using PyTorch and validate it against standard Multi-Head Attention in the degenerate case where `num_heads = 1`.

---

### Requirements

1. **Linear Projections**
   - Q projection: `d_model → d_model` (all query heads)
   - K projection: `d_model → d_head` (single shared head)
   - V projection: `d_model → d_head` (single shared head)
   - Output projection: `d_model → d_model`

2. **Expand K/V to Match Query Heads**
   - Use `.expand()` to broadcast the single K/V head across all query heads.

3. **Scaled Dot-Product Attention**
   - Compute: `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply optional mask before softmax.
   - Weight V with attention weights.

4. **Combine Heads**
   - Concatenate all query head outputs.
   - Apply final linear projection.

5. **Validate Against MHA**
   - When `num_heads = 1`, MQA is identical to single-head MHA.
   - Copy weights and verify outputs match with `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ K and V must be projected to a single head (`d_head`), not `d_model`.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when `num_heads = 1`.

---

<details>
  <summary>💡 Hint</summary>

  - Q shape after projection and reshape: `(batch, num_heads, seq_len, d_head)`
  - K/V shape after projection and reshape: `(batch, 1, seq_len, d_head)`
  - Use `.expand(-1, num_heads, -1, -1)` to broadcast K/V to all heads without copying memory.
  - When `num_heads = 1`, the Q projection is `d_model → d_model` which equals `d_model → 1 * d_head = d_head = d_model`, so all projections have the same shape — identical to single-head MHA.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
class MultiQueryAttention(nn.Module):
    """
    Implements Multi-Query Attention (MQA).
    All query heads share a single key/value head.
    """
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads

        # TODO: Create linear projections
        # Q projects to full d_model (all heads)
        # K and V project to d_head only (single shared head)
        # Output projects d_model -> d_model
        ...

    def forward(self, q, k, v, mask=None):
        """
        Args:
            q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Masking tensor for attention

        Returns:
            Tensor: MQA output of shape (batch_size, seq_len, d_model)
        """
        # TODO: Implement forward pass
        # 1. Project Q, K, V
        # 2. Reshape Q to (batch, num_heads, seq_len, d_head)
        # 3. Reshape K, V to (batch, 1, seq_len, d_head)
        # 4. Expand K, V to (batch, num_heads, seq_len, d_head)
        # 5. Compute scaled dot-product attention
        # 6. Apply mask if provided
        # 7. Combine heads and apply output projection
        ...

In [ ]:
# Validate: when num_heads=1, MQA == single-head MHA
# (because there's only 1 query head and 1 KV head — identical)
num_heads_test = 1

multihead_attn = torch.nn.MultiheadAttention(
    embed_dim=d_model, num_heads=num_heads_test, bias=False, batch_first=True
)

custom_attn = MultiQueryAttention(num_heads_test, d_model)

# Copy weights: when num_heads=1, Q/K/V projections are all d_model -> d_model
custom_attn.Q_w.weight.data = multihead_attn.in_proj_weight[:d_model, :].clone()
custom_attn.K_w.weight.data = multihead_attn.in_proj_weight[d_model:2*d_model, :].clone()
custom_attn.V_w.weight.data = multihead_attn.in_proj_weight[2*d_model:, :].clone()
custom_attn.W_out.weight.data = multihead_attn.out_proj.weight.data.clone()

output_custom = custom_attn(q, k, v)
output_ref, _ = multihead_attn(q, k, v)

print("Custom MQA output:")
print(output_custom)
print("\nPyTorch MHA output (num_heads=1):")
print(output_ref)

assert torch.allclose(output_custom, output_ref, atol=1e-06, rtol=1e-05)
print("\n✅ Outputs match! MQA implementation is correct.")

In [ ]:
# Bonus: verify KV cache savings
# MHA KV cache per token: num_heads * d_head * 2 (K and V)
# MQA KV cache per token: 1 * d_head * 2 (K and V)
print(f"MHA KV cache per token: {num_heads * (d_model // num_heads) * 2} values")
print(f"MQA KV cache per token: {1 * (d_model // num_heads) * 2} values")
print(f"Memory reduction: {num_heads}x")